# LOPO n-plet optimization and evaluation

This notebook runs the full leave-one-pair-out (LOPO) pipeline in two sequential stages:

**Stage 1 — Optimization:** For every conscious / non-responsive scan pair we search,
via simulated annealing, for the set of brain regions (an *n*-plet) whose
O-information difference $\Delta\Omega$ is maximised between the two scans.
We consider two optimisation polarities:
- **$\Omega_C > \Omega_{NR}$**: maximise $\Omega_{\text{conscious}} - \Omega_{\text{non-responsive}}$
- **$\Omega_{NR} > \Omega_C$**: maximise $\Omega_{\text{non-responsive}} - \Omega_{\text{conscious}}$

**Stage 2 — Evaluation:** Each discovered n-plet is scored on all *other* scans
(leave-one-pair-out), computing ANOVA F-score and PR-AUC for
both O-information and a pairwise FC baseline (mean Fisher-*z* correlation).

> **Note:** Stage 1 is GPU-intensive (hours to days depending on hardware).
> If you already have the `R1_A_max_O_diff_*_all_orders.csv` files in `results/`,
> jump directly to Stage 2.

**Note**: This notebook should be run from the `high-order-anesthesia` folder.

In [1]:
from pathlib import Path
import os

def ensure_project_root(target_name: str = "high-order-anesthesia") -> Path:
    cwd = Path.cwd().resolve()
    if cwd.name == target_name:
        return cwd
    for parent in cwd.parents:
        if parent.name == target_name:
            os.chdir(parent)
            return parent
    raise RuntimeError(
        f"Could not find '{target_name}' in current path or parents. "
        f"Please run the notebook from inside the project."
    )

ROOT = ensure_project_root("high-order-anesthesia")
print(f"Now in: {ROOT.name}")

Now in: high-order-anesthesia


## Imports

In [2]:
import pandas as pd
import torch
import numpy as np
import itertools
import logging
import ast
from pathlib import Path
from tqdm.notebook import tqdm

In [3]:
from src.hoi_anesthesia.thoi_utils import simulated_annealing_parallel
from src.hoi_anesthesia.utils import max_difference_pairs, evaluate_nplet_batched
from src.hoi_anesthesia.io import load_covariance_dict, print_time

## Data loading and state definitions

In [4]:
results_path = "results"
data_path    = "data"

all_covs = load_covariance_dict(f"{data_path}/covariance_matrices_gc.h5")

# MA = Multi-Anesthesia dataset,  DBS = Deep Brain Stimulation dataset
conscious_states = {
    "MA":  ["MA_awake"],
    "DBS": ["DBS_awake", "ts_on_5V"],
}
nonresponsive_states = {
    "MA":  ["ts_selv2", "ts_selv4", "moderate_propofol", "deep_propofol", "ketamine"],
    "DBS": ["ts_off", "ts_on_3V_control", "ts_on_5V_control"],
}

---
## Stage 1 — n-plet optimisation (simulated annealing)

For each conscious / non-responsive subject pair we run simulated annealing to
find the n-plet that maximises $\Delta\Omega$ between the two scans.

Two polarities are explored:
- **`c_gt_nr`** ($\Omega_C > \Omega_{NR}$): stacks `[cov_conscious, cov_nr]` so the annealing
  maximises $\Omega_{\text{scan}_1} - \Omega_{\text{scan}_2}$.
- **`nr_gt_c`** ($\Omega_{NR} > \Omega_C$): stacks `[cov_nr, cov_conscious]`.

Results are check-pointed as CSV after every state-pair combination so the run
can be safely interrupted and resumed.

In [7]:
# --- Simulated annealing hyper-parameters ---
early_stop = 1000
max_iter   = 10000
repeat     = 100
batch_size = 50

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.cuda.empty_cache()
print(f"Using device: {device}")

Using device: cuda


In [8]:
datasets_to_optimise = ["MA", "DBS"]
orders = list(range(3, 10))  # 3 to 9

POLARITIES = {
    "c_gt_nr": "c_gt_nr",   # Omega_C > Omega_NR
    "nr_gt_c": "nr_gt_c",   # Omega_NR > Omega_C
}

In [ ]:

for selected_dataset in datasets_to_optimise:
    t_i = __import__("time").time()
    for order in orders:
        results = []
        print("*" * 40)
        print(f"Dataset: {selected_dataset}  |  Order: {order}")

        for state_c, state_nr in itertools.product(
            conscious_states[selected_dataset],
            nonresponsive_states[selected_dataset],
        ):
            covs_c  = all_covs[selected_dataset][state_c]   # (N_c,  82, 82)
            covs_nr = all_covs[selected_dataset][state_nr]  # (N_nr, 82, 82)

            for polarity in ["c_gt_nr", "nr_gt_c"]:
                torch.cuda.empty_cache()
                cov_list       = []
                subject_indices = []

                for i in range(covs_c.shape[0]):
                    for j in range(covs_nr.shape[0]):
                        cov_c  = torch.from_numpy(covs_c[i])
                        cov_nr = torch.from_numpy(covs_nr[j])
                        if polarity == "c_gt_nr":
                            cov_list.append(torch.stack([cov_c, cov_nr], dim=0))
                        else:  # nr_gt_c
                            cov_list.append(torch.stack([cov_nr, cov_c], dim=0))
                        subject_indices.append([i, j])

                X = torch.stack(cov_list, dim=0).to(device)
                n_batches = X.shape[0] // batch_size + 1
                print(
                    f"  {state_c} vs {state_nr}  |  "
                    f"{X.shape[0]} pairs  |  polarity={polarity}"
                )

                for b_idx in range(n_batches):
                    batched_X   = X[b_idx * batch_size : (b_idx + 1) * batch_size]
                    batched_idx = subject_indices[b_idx * batch_size : (b_idx + 1) * batch_size]
                    if batched_X.shape[0] == 0:
                        continue

                    torch.cuda.empty_cache()
                    optimal_nplets, optimal_scores = simulated_annealing_parallel(
                        X=batched_X,
                        order=order,
                        device=device,
                        largest=True,
                        metric=max_difference_pairs,
                        repeat=repeat,
                        early_stop=early_stop,
                        max_iterations=max_iter,
                        covmat_precomputed=True,
                        batch_size=batch_size,
                        verbose=logging.WARNING,
                    )

                    max_idx = torch.argmax(optimal_scores, dim=0)
                    best_nplets = optimal_nplets[max_idx, torch.arange(optimal_nplets.size(1))]
                    best_scores = optimal_scores[max_idx, torch.arange(optimal_scores.size(1))]

                    for score, nplet, sub_idx in zip(best_scores, best_nplets, batched_idx):
                        nplet_sorted = sorted(nplet.tolist())
                        results.append({
                            "order":      order,
                            "polarity":   polarity,
                            "state_c":    state_c,
                            "state_nr":   state_nr,
                            "subject_c":  sub_idx[0],
                            "subject_nr": sub_idx[1],
                            "optimal_nplet": nplet_sorted,
                            "optimal_score": score.item(),
                        })
                    torch.cuda.empty_cache()

                # checkpoint after every state-pair x polarity
                pd.DataFrame(results).to_csv(
                    f"{results_path}/R1_A_max_O_diff_{selected_dataset}_{order}.csv",
                    index=False, encoding="utf-8-sig", sep=";", decimal=",",
                )

        print_time(t_i, __import__("time").time())

### Merge per-order CSVs into a single file per dataset

In [9]:
for selected_dataset in datasets_to_optimise:
    all_orders = pd.concat(
        [
            pd.read_csv(
                Path(results_path) / f"R1_A_max_O_diff_{selected_dataset}_{order}.csv",
                sep=";", decimal=",", encoding="utf-8-sig",
            )
            for order in orders
        ],
        ignore_index=True,
    )
    out = Path(results_path) / f"R1_A_max_O_diff_{selected_dataset}_all_orders.csv"
    all_orders.to_csv(out, index=False, encoding="utf-8-sig", sep=";", decimal=",")
    print(f"Saved: {out}  ({len(all_orders)} rows)")

Saved: results\R1_A_max_O_diff_MA_all_orders.csv  (31920 rows)
Saved: results\R1_A_max_O_diff_DBS_all_orders.csv  (81130 rows)


---
## Stage 2 — n-plet evaluation (LOPO PR-AUC)

Each discovered n-plet is evaluated on **all scans except the discovery pair**
(leave-one-pair-out holdout). For each n-plet and scan we compute:

- HOI measures: TC, DTC, $\Omega$, S
- Pairwise FC baseline: mean Fisher-*z*-transformed Pearson correlation within the n-plet

Discrimination quality is quantified by ANOVA F-score and PR-AUC.
Results are check-pointed every 10 000 rows.

In [10]:
# Load optimisation results
results_dict = {
    ds: pd.read_csv(
        f"{results_path}/R1_A_max_O_diff_{ds}_all_orders.csv",
        encoding="utf-8-sig", sep=";", decimal=",",
    )
    for ds in ["MA", "DBS"]
}

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.cuda.empty_cache()

for ds, df in results_dict.items():
    orders = sorted(df['order'].unique().astype(int).tolist())
    print(f"{ds}: {len(df)} n-plets | orders: {orders}")


MA: 31920 n-plets | orders: [3, 4, 5, 6, 7, 8, 9]
DBS: 81130 n-plets | orders: [3, 4, 5, 6, 7, 8, 9]


In [ ]:
for dataset_name, results_df in results_dict.items():
    eval_results = []

    for idx, row in tqdm(
        results_df.iterrows(),
        total=len(results_df),
        desc=f"Evaluating {dataset_name}",
    ):
        metrics = evaluate_nplet_batched(
            idx,
            all_covs,
            conscious_states,
            nonresponsive_states,
            dataset_name,
            ast.literal_eval(row["optimal_nplet"]),
            row["state_c"],
            row["state_nr"],
            row["subject_c"],
            row["subject_nr"],
            device=device,
        )
        eval_results.extend(metrics)

        # checkpoint every 10k rows
        if idx % 10_000 == 0 and idx > 0:
            metrics_df = pd.DataFrame(eval_results)
            out_df = results_df.merge(
                metrics_df, left_index=True, right_on="row_idx"
            ).drop(columns=["row_idx"])
            out_df.to_csv(
                f"{results_path}/R1_A_nplet_eval_{dataset_name}.csv",
                index=False, encoding="utf-8-sig", sep=";", decimal=",",
            )

    # final save
    metrics_df = pd.DataFrame(eval_results)
    out_df = results_df.merge(
        metrics_df, left_index=True, right_on="row_idx"
    ).drop(columns=["row_idx"])
    out_df.to_csv(
        f"{results_path}/R1_A_nplet_eval_{dataset_name}.csv",
        index=False, encoding="utf-8-sig", sep=";", decimal=",",
    )
    print(f"Saved R1_A_nplet_eval_{dataset_name}.csv  ({len(out_df)} rows)")

### Inspect best n-plets

In [ ]:
# TEMPORARY: Load and reformat pre-computed evaluation results
eval_MA  = pd.read_csv(f"{results_path}/A_2_C_nplet_eval_MA.csv",  sep=";", decimal=",", encoding="utf-8-sig")
eval_DBS = pd.read_csv(f"{results_path}/A_2_C_nplet_eval_DBS.csv", sep=";", decimal=",", encoding="utf-8-sig")

TASK_TO_POLARITY = {"C_vs_NR": "c_gt_nr", "NR_vs_C": "nr_gt_c"}

for ds_name, eval_df in [("MA", eval_MA), ("DBS", eval_DBS)]:
    eval_df = eval_df.rename(columns={"task": "polarity"})
    eval_df["polarity"] = eval_df["polarity"].map(TASK_TO_POLARITY)
    # Column order matching Stage 2 output
    cols = ["order", "polarity", "state_c", "state_nr", "subject_c", "subject_nr",
            "optimal_nplet", "optimal_score", "measure", "F_score", "PR_AUC", "PR_AUC_inv"]
    eval_df = eval_df[cols]
    eval_df.to_csv(
        f"{results_path}/R1_A_nplet_eval_{ds_name}.csv",
        index=False, encoding="utf-8-sig", sep=";", decimal=",",
    )
    if ds_name == "MA":
        eval_MA = eval_df
    else:
        eval_DBS = eval_df

print("Done — task renamed to polarity, C_vs_NR -> c_gt_nr, NR_vs_C -> nr_gt_c")

Done — task renamed to polarity, C_vs_NR -> c_gt_nr, NR_vs_C -> nr_gt_c


In [ ]:
# Load final evaluation results
eval_MA  = pd.read_csv(f"{results_path}/R1_A_nplet_eval_MA.csv",  sep=";", decimal=",", encoding="utf-8-sig")
eval_DBS = pd.read_csv(f"{results_path}/R1_A_nplet_eval_DBS.csv", sep=";", decimal=",", encoding="utf-8-sig")

print(eval_MA.head(3))

   order polarity   state_c  state_nr  subject_c  subject_nr optimal_nplet  \
0      2  c_gt_nr  MA_awake  ts_selv2          0           0      [42, 72]   
1      2  c_gt_nr  MA_awake  ts_selv2          0           0      [42, 72]   
2      2  c_gt_nr  MA_awake  ts_selv2          0           0      [42, 72]   

   optimal_score measure   F_score    PR_AUC  PR_AUC_inv  
0   8.881784e-16      TC  0.102216  0.227073    0.201486  
1   8.881784e-16     DTC  0.102216  0.227073    0.201486  
2   8.881784e-16       O  0.000000  0.211039    0.196021  


In [27]:
# Best n-plets for Omega_C > Omega_NR polarity (c_gt_nr)
(
    eval_MA
    .query("measure == 'O' and polarity == 'c_gt_nr'")
    .sort_values("PR_AUC", ascending=False)
    .head(2)
)

,order,polarity,state_c,state_nr,subject_c,subject_nr,optimal_nplet,optimal_score,measure,F_score,PR_AUC,PR_AUC_inv
166010,8,c_gt_nr,MA_awake,ts_selv2,17,2,"[7, 14, 23, 26, 35, 55, 67, 76]",3.739858,O,393.258988,0.996298,0.109725
166034,8,c_gt_nr,MA_awake,ts_selv2,17,6,"[7, 14, 23, 31, 55, 72, 80, 81]",3.619432,O,472.517163,0.996298,0.109725


In [28]:
# Best n-plets for Omega_NR > Omega_C polarity (nr_gt_c)
(
    eval_MA
    .query("measure == 'O' and polarity == 'nr_gt_c'")
    .sort_values("PR_AUC_inv", ascending=False)
    .head(2)
)

,order,polarity,state_c,state_nr,subject_c,subject_nr,optimal_nplet,optimal_score,measure,F_score,PR_AUC,PR_AUC_inv
79412,4,nr_gt_c,MA_awake,ketamine,3,17,"[38, 40, 45, 81]",0.576925,O,142.231478,0.109898,0.970251
29966,3,nr_gt_c,MA_awake,ts_selv2,0,2,"[23, 34, 81]",0.508705,O,160.040724,0.111550,0.943832


---
### Collapsed n-plets (aggregated across state pairs and subjects)

In [29]:
group_cols = ["order", "polarity", "optimal_nplet", "measure"]
agg_cols   = ["optimal_score", "F_score", "PR_AUC", "PR_AUC_inv"]

collapsed = {}
for ds_name, eval_df in [("MA", eval_MA), ("DBS", eval_DBS)]:
    grp = eval_df.groupby(group_cols)[agg_cols]
    mean_df   = grp.mean().add_suffix("_mean")
    median_df = grp.median().add_suffix("_median")
    collapsed[ds_name] = pd.concat([mean_df, median_df], axis=1).reset_index()

collapsed_MA  = collapsed["MA"]
collapsed_DBS = collapsed["DBS"]
print(f"MA collapsed:  {len(collapsed_MA)} rows | DBS collapsed: {len(collapsed_DBS)} rows")

MA collapsed:  62892 rows | DBS collapsed: 180750 rows


In [33]:
# Best n-plet per dataset × polarity ranked by mean PR_AUC (measure == 'O')
# c_gt_nr sorted by PR_AUC_mean, nr_gt_c sorted by PR_AUC_inv_mean
for ds_name, cdf in [("MA", collapsed_MA), ("DBS", collapsed_DBS)]:
    print(f"\n=== {ds_name} — best by mean PR_AUC ===")
    o_df = cdf.query("measure == 'O'")
    best = pd.concat([
        o_df[o_df["polarity"] == "c_gt_nr"].sort_values("PR_AUC_mean",     ascending=False).head(1),
        o_df[o_df["polarity"] == "nr_gt_c"].sort_values("PR_AUC_inv_mean", ascending=False).head(1),
    ]).reset_index(drop=True)
    display(best[["polarity", "order", "optimal_nplet", "PR_AUC_mean", "PR_AUC_inv_mean", "F_score_mean"]])


=== MA — best by mean PR_AUC ===


,polarity,order,optimal_nplet,PR_AUC_mean,PR_AUC_inv_mean,F_score_mean
0,c_gt_nr,9,"[7, 14, 23, 27, 31, 55, 72, 80, 81]",0.996298,0.109725,449.346466
1,nr_gt_c,4,"[38, 40, 45, 81]",0.109898,0.970251,142.231478



=== DBS — best by mean PR_AUC ===


,polarity,order,optimal_nplet,PR_AUC_mean,PR_AUC_inv_mean,F_score_mean
0,c_gt_nr,9,"[8, 13, 18, 21, 22, 33, 42, 61, 74]",0.971319,0.303688,112.769114
1,nr_gt_c,3,"[3, 4, 10]",0.509672,0.718789,5.247227
